# 01 · Setup & Data (Colab)

在 Colab 跑这个 notebook：拉本地代码 → Kaggle 认证 → 下载数据 → 验证 import。

运行前：菜单 **Runtime → Change runtime type → T4 GPU**。

## Cell 1 · 拉本地代码 (从 GitHub)

In [ ]:
# 先删旧 clone，保证每次重跑都拿到最新代码 + 正确分支（避免 Colab 残留旧版）
!rm -rf Kaggle-Study-
# -b 后跟你正在开发的分支（当前 process-data；以后合并到 main 后改成 main）
!git clone -b process-data https://github.com/SleepyEveryD/Kaggle-Study-.git
import sys
sys.path.append('/content/Kaggle-Study-/IAM-Handwritten-Forms-Dataset')

# 核对：应打印 process-data + 最新 commit（和本地一致）
!cd Kaggle-Study- && git branch --show-current && git log --oneline -1

# 注：改了 src 后重跑，最稳妥 Runtime → Restart session（避免 Python 模块缓存旧版）

## Cell 2 · Kaggle 认证

推荐用左侧 🔑 Secrets 建 `KAGGLE_USERNAME` / `KAGGLE_KEY`（并打开 Notebook access）。

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

# —— 或最简版（明文，仅自己用、别分享/别提交）：
# os.environ['KAGGLE_USERNAME'] = '你的用户名'
# os.environ['KAGGLE_KEY']      = '你复制的key'

!pip -q install kaggle

## Cell 3 · 下载数据（Google Drive 持久化）

策略：**只从 Kaggle 下载一次** → 把单个 zip 缓存到 Drive；**每次会话**从 Drive 解压到本地 `/content`（训练读本地才快，别直接读 Drive 上的海量小文件）。

首次运行会弹出 Drive 授权窗口，点同意即可。

In [ ]:
import os, subprocess, shutil

# 挂载 Google Drive（首次弹授权窗口，点同意）
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/iam-handwritten-forms'   # Drive 持久化目录
ZIP_NAME  = 'iam-handwritten-forms-dataset.zip'
DRIVE_ZIP = os.path.join(DRIVE_DIR, ZIP_NAME)
LOCAL_DIR = '/content/data/iam'                              # 本次会话本地解压目录
os.makedirs(DRIVE_DIR, exist_ok=True)

# 1) Drive 上没有 zip → 从 Kaggle 下载一次，存进 Drive
if not os.path.exists(DRIVE_ZIP):
    print('⬇️ Drive 无缓存，从 Kaggle 下载...')
    subprocess.run('kaggle datasets download -d naderabdelghany/iam-handwritten-forms-dataset '
                   '-p /content/tmp', shell=True, check=True)
    shutil.move(f'/content/tmp/{ZIP_NAME}', DRIVE_ZIP)
    print('✅ 已缓存到 Drive：', DRIVE_ZIP)
else:
    print('✅ Drive 已有缓存，跳过 Kaggle 下载：', DRIVE_ZIP)

# 2) 本地还没解压 → 从 Drive 的 zip 解压到本地（每次会话；本地读训练才快）
if os.path.isdir(f'{LOCAL_DIR}/data') and os.listdir(f'{LOCAL_DIR}/data'):
    print('✅ 本地已解压，跳过')
else:
    print('📦 从 Drive 解压到本地 /content ...')
    os.makedirs(LOCAL_DIR, exist_ok=True)
    subprocess.run(f'unzip -q -o "{DRIVE_ZIP}" -d {LOCAL_DIR}', shell=True, check=True)
    print('✅ 解压完成')

!ls {LOCAL_DIR}

## Cell 3.5 · 探查真实数据结构

真实数据在里层 `/content/data/iam/data/`。一次问清：标签在不在、由什么文件组成、命名规律。

In [ ]:
import os
ROOT  = '/content/data/iam'
INNER = f'{ROOT}/data'

# A) 一个样本编号文件夹里有什么
sample = sorted(os.listdir(INNER))[0]
print(f'--- 样本文件夹 {INNER}/{sample} 内容（前 25 个）---')
!ls -laR {INNER}/{sample} | head -30

# B) 全树里所有非图片文件 —— 标签最可能藏在这里 (csv/json/txt/xml/ipynb)
print('\n--- 全树非图片文件（找标签！）---')
!find {ROOT} -type f ! -iname "*.png" ! -iname "*.jpg" ! -iname "*.jpeg" ! -iname "*.tif" | head -50

# C) 各扩展名文件数量统计
print('\n--- 文件扩展名统计 ---')
!find {ROOT} -type f | sed 's/.*\\.//' | sort | uniq -c | sort -rn | head -20

# D) 图片完整路径样例
print('\n--- 图片路径样例 ---')
!find {ROOT} -iname "*.png" -o -iname "*.jpg" -o -iname "*.tif" | head -8

## Cell 4 · 验证能 import 本地代码 + 路径就绪

In [ ]:
from src import config
print('ENV =', config.ENV)
print('DATA_DIR =', config.DATA_DIR)